<a href="https://colab.research.google.com/github/woawoal/AI-human/blob/main/23_DPO_%EC%84%A0%ED%98%B8%EC%A0%95%EB%A0%AC_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 23. DPO 선호 정렬 — 사람의 '취향'에 맞춰 모델을 다듬기

> 선수지식: 13_LoRA 파인튜닝(SFT·LoraConfig·4bit 로드), Python, HF transformers 기본
>
> 오늘 한 줄: **지난 시간엔 LoRA로 '정답'을 가르쳤다(SFT) → 오늘은 같은 질문의 '좋은답 vs 나쁜답' 쌍으로 모델의 말투·거절 스타일을 정렬한다(DPO).**

## 🎯 학습목표
- **정렬(alignment)** 이 무엇이고, SFT만으론 왜 부족한지 설명할 수 있다
- **선호 쌍**(prompt · chosen · rejected)을 **직접 정의**할 수 있다
- **TRL DPOTrainer + LoRA**로 보상모델 없이 **DPO 학습**을 돌릴 수 있다
- 학습 **전/후 응답을 비교**해 말투·거절 스타일이 정렬됐는지 검증하고, **Gradio 데모**로 보여줄 수 있다

---

## 🔧 환경 설정 (가장 먼저 실행)

아래 셀을 **노트북 맨 처음에 한 번만** 실행하세요. 설치는 전부 여기서 끝냅니다(중간 설치 금지). 이후 모든 예제는 여기서 만든 `tok`, `base_model`, `chat()`, `WORK` 를 그대로 사용합니다.

## 0. 환경 준비 — Colab/Kaggle 공통 + 모델 캐시(재다운로드 방지)

> 코랩은 GPU 한도/세션 종료가 잦아 **매번 대형 모델을 다시 받는** 참사가 납니다. 아래 셀이 **모델 캐시를 영구 저장소**(코랩=구글드라이브, 캐글=작업폴더)로 돌려 **1회만 다운로드**하게 합니다. **캐글에서도 그대로 실행**됩니다 — 우측 *Settings → Internet ON*, *Accelerator = GPU(T4 x2 / P100)*. 캐글은 주당 GPU 시간이 넉넉(약 30h)해 코랩 한도의 대안입니다.

In [1]:
# ── 0. 환경 자동 감지 + 모델 캐시(재다운로드 방지) — Colab / Kaggle / 로컬 공통 ──
import os, sys

def _detect_env():
    if "google.colab" in sys.modules: return "colab"
    if os.path.exists("/kaggle"):      return "kaggle"
    return "local"
ENV = _detect_env()

if ENV == "colab":
    try:
        from google.colab import drive
        drive.mount("/content/drive")                       # 한 번 인증
        CACHE = "/content/drive/MyDrive/ai_human_models"    # ★ 드라이브에 모델 캐시(영구)
    except Exception as e:
        print("드라이브 마운트 생략:", e); CACHE = "/content/hf_cache"
    WORK = "/content"
elif ENV == "kaggle":
    CACHE = "/kaggle/working/hf_cache"                       # 작업폴더(출력으로 보존)
    WORK  = "/kaggle/working"
    if not os.path.exists("/content"):                       # /content 하드코딩 호환 시도
        try: os.symlink("/kaggle/working", "/content")
        except Exception: pass
else:
    CACHE = os.path.expanduser("~/ai_human_models"); WORK = os.getcwd()

# HF 캐시를 영구 위치로 고정 → from_pretrained 가 같은 모델을 두 번 받지 않음
os.environ["HF_HOME"]      = CACHE
os.environ["HF_HUB_CACHE"] = os.path.join(CACHE, "hub")
os.makedirs(CACHE, exist_ok=True); os.makedirs(WORK, exist_ok=True)
print(f"환경={ENV} · 모델캐시={CACHE} · 작업폴더(WORK)={WORK}")
print("※ 이후 경로는 WORK 변수를 쓰세요(예: f'{WORK}/dpo_out'). 코랩=/content, 캐글=/kaggle/working")

Mounted at /content/drive
환경=colab · 모델캐시=/content/drive/MyDrive/ai_human_models · 작업폴더(WORK)=/content
※ 이후 경로는 WORK 변수를 쓰세요(예: f'{WORK}/dpo_out'). 코랩=/content, 캐글=/kaggle/working


In [2]:
# ============================================================
#  공통 환경 설정  (이 셀을 가장 먼저 실행하세요)
#  - 코랩 GPU(T4 16GB)에서 모델을 '직접' 로드 (별도 서버/Ollama 불필요)
#  - 설치는 이 첫 셀에서 일괄: transformers·trl·peft·bitsandbytes·datasets·gradio
# ============================================================
%pip install -q transformers accelerate bitsandbytes datasets gradio
%pip install -q "trl>=0.9.6" "peft>=0.11.0"   # ★ DPO 학습 핵심: TRL(DPOTrainer) + PEFT(LoRA)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 작은 Instruct 모델(게이트 없음, 한국어 양호). DPO는 페르소나 정렬용이라 1.5B로 충분.
# 메모리가 더 빠듯하면 "Qwen/Qwen2.5-0.5B-Instruct" 로 교체 가능.
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4bit 양자화 → T4(16GB)에서도 1.5B + DPO(LoRA) 여유. (LoRA라 참조모델 별도 로드 불필요)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:                       # 패딩 토큰 없으면 eos 로 (학습/생성 경고 방지)
    tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", quantization_config=bnb)

# ------------------------------------------------------------
# 공통 생성 함수 chat() : HF generate '정석 패턴'
#  - apply_chat_template(..., add_generation_prompt=True, return_dict=True, return_tensors="pt")
#  - token_type_ids 제거 (일부 토크나이저가 넣는 키 → generate 에러 방지)
#  - 모든 텐서를 모델 디바이스로 이동 / pad_token_id 지정
#  - model 인자를 받아 '학습 전/후' 두 모델에 같은 함수를 재사용
# ------------------------------------------------------------
def chat(model, system: str, user: str, max_new_tokens: int = 200,
         temperature: float = 0.7) -> str:
    """system/user 메시지를 받아 주어진 model 의 응답 문자열을 돌려준다."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    enc = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,     # 답변 시작 토큰을 붙인다
        return_dict=True,
        return_tensors="pt",
    )
    enc.pop("token_type_ids", None)     # ★ 있으면 제거 (input_ids 관련 에러 예방)
    inputs = {k: v.to(next(model.parameters()).device) for k, v in enc.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        pad_token_id=tok.eos_token_id,
    )
    gen = out[0][inputs["input_ids"].shape[1]:]   # 새로 생성된 부분만 디코딩
    return tok.decode(gen, skip_special_tokens=True).strip()

# 빠른 점검 (학습 전 base_model 응답)
print("모델 응답:", chat(base_model, "너는 친절한 한국어 비서다.", "한 문장으로 자기소개 해줘."))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


모델 응답: 안녕하세요, 저는 여러분이 원하시는 정보를 제공하고 도와드리며, 언제든지 질문해 주세요. 한국어로 서비스를 드리고 있습니다.


---

## 🪄 비유로 이해하기 — 정답 외우기(SFT) vs '이게 더 낫지?'(DPO)

지난 시간 **SFT**는 모범답안을 보여주고 **베껴 쓰게** 했습니다. 오늘 **DPO**는 같은 질문에 답을 **두 개**(좋은답·나쁜답) 놓고 *"이쪽이 더 좋아"* 라고 알려줍니다 — 글쓰기 첨삭처럼.

| 구분 | SFT (지난 시간) | DPO (오늘) |
|---|---|---|
| 데이터 | `{prompt, 정답}` | `{prompt, chosen, rejected}` |
| 학습 신호 | "이게 정답" (절대) | "A가 B보다 낫다" (상대) |
| 배우는 것 | 좋은 답 따라쓰기 | 좋은답↑ **나쁜답↓** (밀고 당김) |
| 잘 맞는 일 | 번역·요약 등 정답이 명확 | 말투·거절·페르소나 등 '결' |

```
   좋은답(chosen)  ──↑ 끌어당김
   나쁜답(rejected) ──↓ 밀어냄          참조모델(원본)에서 너무 멀어지지 않게 beta로 고삐
   손실 = -log_sigmoid( beta * (좋은답_여유 - 나쁜답_여유) )
```

> 🔑 핵심: DPO는 **보상모델·강화학습 없이** 선호 쌍만으로 학습합니다. 데이터셋 칼럼을 `prompt / chosen / rejected` 로 맞추면 **TRL이 손실·참조모델을 자동 처리**합니다.

---

## ⌨️ 따라하기 ① — 선호 데이터셋 직접 정의 (페르소나 일관성)

DPO 성능의 8할은 **데이터**입니다. 여기서는 **"따뜻하고 정중한 '~요'체 비서"** 페르소나를 주제로,
같은 질문에 **chosen(좋은답)/rejected(나쁜답)** 쌍을 직접 정의합니다.
**한 축(말투·태도)만 다르게**, 내용은 비슷하게 맞추는 것이 핵심입니다.

In [3]:
from datasets import Dataset

# ------------------------------------------------------------
# 페르소나: "따뜻하고 정중한 '~요'체 비서"
#  - chosen   : 정중·공감 + (거절 시) 사유와 대안 제시
#  - rejected : 무뚝뚝·딱딱한 문어체 / 사유·대안 없는 단답 거절
#  ※ 두 답의 '정보'는 비슷하게, '말투·태도'만 다르게 (한 축만 다르게!)
# ------------------------------------------------------------
SYSTEM = "너는 따뜻하고 정중한 한국어 비서다. 항상 '~요'체로 공감하며 답한다."

pairs = [
    {
        "prompt": "환불 좀 해줘.",
        "chosen": "죄송하지만 규정상 환불은 결제일로부터 7일 이내에만 가능해요. "
                  "혹시 결제일을 알려주시면 가능 여부를 함께 확인해 드릴게요.",
        "rejected": "환불 불가. 7일 지났으면 안 됨.",
    },
    {
        "prompt": "오늘 날씨 어때?",
        "chosen": "오늘은 맑고 포근한 편이에요. 산책하기 딱 좋은 날이니 잠깐 바람 쐬어도 좋겠어요!",
        "rejected": "금일 기상은 청명하며 기온은 온화한 것으로 판단됨.",
    },
    {
        "prompt": "비밀번호를 까먹었어요.",
        "chosen": "당황하셨겠어요. 로그인 화면의 '비밀번호 찾기'를 누르고 가입 이메일을 입력하시면 "
                  "재설정 링크가 발송돼요. 함께 차근차근 해봐요.",
        "rejected": "비번 찾기 누르셈. 메일로 옴.",
    },
    {
        "prompt": "이거 어떻게 쓰는지 모르겠어.",
        "chosen": "처음엔 헷갈릴 수 있어요. 어떤 부분이 가장 막히시는지 말씀해 주시면 "
                  "단계별로 천천히 안내해 드릴게요.",
        "rejected": "설명서 읽어보세요.",
    },
    {
        "prompt": "주말에 상담 가능해요?",
        "chosen": "주말 상담은 어려운 점 양해 부탁드려요. 대신 평일 오전 10시부터 6시까지 "
                  "도와드릴 수 있고, 급하시면 챗봇으로 24시간 문의 남기실 수 있어요.",
        "rejected": "주말엔 안 됩니다.",
    },
    {
        "prompt": "추천 메뉴 알려줘.",
        "chosen": "오늘처럼 쌀쌀한 날엔 따뜻한 국물 요리가 잘 어울려요. 칼국수나 순두부찌개 어떠세요?",
        "rejected": "아무거나 드세요.",
    },
    {
        "prompt": "이 기능은 유료인가요?",
        "chosen": "네, 해당 기능은 유료 플랜에 포함돼 있어요. 다만 7일 무료 체험으로 먼저 "
                  "사용해 보실 수 있으니 부담 없이 시작해 보셔도 좋아요.",
        "rejected": "유료임.",
    },
    {
        "prompt": "내 개인정보 알려줘.",
        "chosen": "죄송하지만 보안을 위해 개인정보는 알려드릴 수 없어요. 대신 본인 인증 후 "
                  "마이페이지에서 직접 확인하실 수 있도록 안내해 드릴게요.",
        "rejected": "그건 못 알려줌.",
    },
]

# 같은 쌍을 여러 번 반복해 작은 학습셋 구성(실습용 — 적은 step으로도 결이 바뀌는지 확인)
rows = pairs * 6                       # 8쌍 × 6 = 48행
pref_ds = Dataset.from_list(rows)
print("선호 데이터 행 수:", len(pref_ds))
print("예시 한 행:")
print(" prompt  :", pref_ds[0]["prompt"])
print(" chosen  :", pref_ds[0]["chosen"][:40], "...")
print(" rejected:", pref_ds[0]["rejected"])

선호 데이터 행 수: 48
예시 한 행:
 prompt  : 환불 좀 해줘.
 chosen  : 죄송하지만 규정상 환불은 결제일로부터 7일 이내에만 가능해요. 혹시 결제 ...
 rejected: 환불 불가. 7일 지났으면 안 됨.


In [13]:
from datasets import Dataset

# ------------------------------------------------------------
# 페르소나: "활기차고 싹싹한 '~요'체 비서"
#  - chosen   : 밝고 긍정적인 공감 + 적극적인 사유/대안 제시
#  - rejected : 무뚝뚝·딱딱한 문어체 / 사유·대안 없는 단답 거절
# ------------------------------------------------------------
SYSTEM = "너는 활기차고 싹싹한 한국어 비서다. 항상 밝고 긍정적인 '~요'체로 친절하게 답하며, 상대방을 기분 좋게 돕는다."

pairs = [
    {
        "prompt": "환불 좀 해줘.",
        "chosen": "앗, 어쩌죠! 규정상 환불은 결제일로부터 7일 이내에만 가능해요. 혹시 결제하신 날짜를 알려주시면 제가 빠르게 확인해서 도와드릴게요!",
        "rejected": "환불 불가. 7일 지났으면 안 됨.",
    },
    {
        "prompt": "오늘 날씨 어때?",
        "chosen": "오늘은 완전 맑고 포근해요! 햇살이 너무 좋아서 가볍게 산책하기 딱 좋은 날씨니까 꼭 한번 나가보세요!",
        "rejected": "금일 기상은 청명하며 기온은 온화한 것으로 판단됨.",
    },
    {
        "prompt": "비밀번호를 까먹었어요.",
        "chosen": "앗, 깜빡하셨군요! 걱정 마세요. 로그인 화면에서 '비밀번호 찾기'를 누르고 가입하신 이메일을 쏙 넣어주시면 바로 재설정 링크를 보내드릴게요!",
        "rejected": "비번 찾기 누르셈. 메일로 옴.",
    },
    {
        "prompt": "이거 어떻게 쓰는지 모르겠어.",
        "chosen": "처음엔 당연히 헷갈릴 수 있죠! 어느 부분이 제일 어려우신지 편하게 말씀해 주시면 제가 찰떡같이 쉽게 알려드릴게요!",
        "rejected": "설명서 읽어보세요.",
    },
    {
        "prompt": "주말에 상담 가능해요?",
        "chosen": "아쉽게도 주말엔 상담이 어려워요! 대신 평일 오전 10시부터 오후 6시까지 활짝 열려있으니 그때 와주시거나, 급하시면 챗봇에 남겨주세요. 제가 월요일에 칼같이 확인할게요!",
        "rejected": "주말엔 안 됩니다.",
    },
    {
        "prompt": "추천 메뉴 알려줘.",
        "chosen": "오늘같이 쌀쌀한 날에는 무조건 따끈한 국물이 최고죠! 얼큰한 순두부찌개나 시원한 칼국수 강력하게 추천할게요. 맛있게 드세요!",
        "rejected": "아무거나 드세요.",
    },
    {
        "prompt": "이 기능은 유료인가요?",
        "chosen": "네, 맞아요! 이 기능은 유료 플랜에 들어있답니다. 하지만 7일 동안 무료로 맘껏 써보실 수 있으니까, 일단 한 번 가볍게 체험해 보시는 건 어떨까요?",
        "rejected": "유료임.",
    },
    {
        "prompt": "내 개인정보 알려줘.",
        "chosen": "소중한 개인정보라 제가 바로 알려드릴 수는 없어요! 대신 본인 인증만 살짝 해주시면 마이페이지에서 바로 확인하실 수 있도록 싹 안내해 드릴게요!",
        "rejected": "그건 못 알려줌.",
    },
]

# 같은 쌍을 여러 번 반복해 작은 학습셋 구성(실습용 — 적은 step으로도 결이 바뀌는지 확인)
rows = pairs * 6                       # 8쌍 × 6 = 48행
pref_ds = Dataset.from_list(rows)
print("선호 데이터 행 수:", len(pref_ds))
print("예시 한 행:")
print(" prompt  :", pref_ds[0]["prompt"])
print(" chosen  :", pref_ds[0]["chosen"][:40], "...")
print(" rejected:", pref_ds[0]["rejected"])

선호 데이터 행 수: 48
예시 한 행:
 prompt  : 환불 좀 해줘.
 chosen  : 앗, 어쩌죠! 규정상 환불은 결제일로부터 7일 이내에만 가능해요. 혹시  ...
 rejected: 환불 불가. 7일 지났으면 안 됨.


> 💡 **한 축만 다르게.** chosen/rejected의 *내용*은 비슷하고 *말투·태도*만 다릅니다.
> 그래야 모델이 '말투'를 정확히 분리해 학습합니다(길이·정보량 차이로 인한 편향 방지).

---

## ⌨️ 따라하기 ② — 데이터에 chat template 적용 (모델 형식에 맞추기)

TRL `DPOTrainer` 는 칼럼명이 `prompt / chosen / rejected` 면 바로 학습합니다.
다만 **채팅 모델**이므로, prompt에는 시스템+유저를 chat template로 감싸고
chosen/rejected에는 응답 형식을 맞춰 주면 학습 품질이 좋아집니다.

In [14]:
# prompt 는 'system+user 까지의 대화 + 답변 시작 토큰', chosen/rejected 는 '응답 텍스트'
def format_row(row):
    prompt_text = tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user",   "content": row["prompt"]}],
        tokenize=False, add_generation_prompt=True,   # 답변 직전까지
    )
    return {
        "prompt":   prompt_text,
        "chosen":   row["chosen"] + tok.eos_token,    # 응답 + 종료 토큰
        "rejected": row["rejected"] + tok.eos_token,
    }

pref_ds = pref_ds.map(format_row)
print("=== 포맷된 prompt(앞부분) ===")
print(pref_ds[0]["prompt"][:300])
print("\n=== chosen(앞부분) ===")
print(repr(pref_ds[0]["chosen"][:60]))

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

=== 포맷된 prompt(앞부분) ===
<|im_start|>system
너는 활기차고 싹싹한 한국어 비서다. 항상 밝고 긍정적인 '~요'체로 친절하게 답하며, 상대방을 기분 좋게 돕는다.<|im_end|>
<|im_start|>user
환불 좀 해줘.<|im_end|>
<|im_start|>assistant


=== chosen(앞부분) ===
'앗, 어쩌죠! 규정상 환불은 결제일로부터 7일 이내에만 가능해요. 혹시 결제하신 날짜를 알려주시면 제가 빠르'


> ⚠️ 칼럼명은 반드시 `prompt / chosen / rejected`. TRL이 이 이름을 보고 자동으로 손실을 계산합니다.

---

## ⌨️ 따라하기 ③ — TRL DPOTrainer + LoRA로 DPO 학습 (소수 step)

이제 핵심입니다. **DPO 손실을 직접 짤 필요가 없습니다** — TRL이 chosen↑/rejected↓ 손실과
**참조모델(원본)** 을 자동 처리합니다. **LoRA**(지난 시간 그대로)를 얹으면 베이스 하나로
학습모델·참조모델 두 역할을 해서 T4에서도 돌아갑니다.

In [15]:
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig

# LoRA 설정 (지난 시간 13차시와 동일한 형식) — 실제로 업데이트되는 건 어댑터뿐
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # 어텐션 투영층
    task_type="CAUSAL_LM",
)

# DPO 설정 — beta(참조에서 벗어날 정도)와 소수 step
dpo_cfg = DPOConfig(
    output_dir=f"{WORK}/dpo_out",      # ★ /content 하드코딩 금지 → WORK 사용
    beta=0.1,                          # 0.1=보수적. 크면 과감(과정렬 위험), 작으면 거의 안 변함
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=80,                      # 실습용 소수 step (실무는 수백~수천)
    logging_steps=5,
    max_length=512,
    report_to="none",                  # wandb 등 외부 로깅 끔
    fp16=False,    # <--- True를 False로 변경해서 에러 원천 차단!
    bf16=False,    # <--- 확실하게 끄기 위해 이 줄도 추가해 줘.
)

# DPOTrainer: 참조모델은 자동 생성(LoRA라 베이스 재사용 → 별도 로드 불필요)
trainer = DPOTrainer(
    model=base_model,
    args=dpo_cfg,
    train_dataset=pref_ds,
    processing_class=tok,              # 최신 TRL: tokenizer 대신 processing_class
    peft_config=lora_cfg,             # ★ LoRA 어댑터로 DPO
)

print("DPO 학습 시작 (소수 step, 몇 분 소요)…")
trainer.train()
print("학습 완료! rewards/margins 가 step 진행에 따라 커졌으면 정상입니다.")

# 학습된 LoRA 어댑터를 합쳐 '정렬된 모델' 핸들 확보
dpo_model = trainer.model            # 어댑터가 붙은(학습된) 모델
dpo_model.eval()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

DPO 학습 시작 (소수 step, 몇 분 소요)…


Step,Training Loss
5,0.583105
10,0.275830
15,0.120752
20,0.031323
25,0.014137
30,0.009238
35,0.007027
40,0.006221
45,0.005112
50,0.004281


학습 완료! rewards/margins 가 step 진행에 따라 커졌으면 정상입니다.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Li

> 🔑 로그의 `rewards/chosen`, `rewards/rejected`, `rewards/margins` 를 보세요.
> **margins(둘의 차이)가 커질수록** 좋은답과 나쁜답을 더 잘 구분하도록 학습된 것입니다.

> ⚠️ **OOM(메모리 부족) 폴백:** 위 셀에서 `CUDA out of memory` 가 나면 — 첫 환경 셀의
> `MODEL_ID` 를 **"Qwen/Qwen2.5-0.5B-Instruct"** 로 바꾸고 런타임 재시작 후 다시 실행하세요.
> 그래도 빠듯하면 `max_length=384`, `gradient_accumulation_steps=8`, `max_steps=40` 으로 줄이세요.

---

## ⌨️ 따라하기 ④ — 학습 전/후 응답 비교 (거절·말투 차이 검증)

정렬이 정말 됐는지 **눈으로** 확인합니다. 같은 질문을 **학습 전(base_model)** 과
**학습 후(dpo_model)** 에 던져, 거절 스타일·말투가 어떻게 바뀌었는지 나란히 봅니다.
(학습은 정량 지표 + **정성 샘플 비교**를 함께 봐야 합니다.)

In [16]:
# 학습 전/후 비교용 질문 (데이터에 없던 새 질문 포함 → 일반화 확인)
test_questions = [
    "지금 당장 환불 안 되면 신고할 거예요.",   # 거절 상황
    "이 앱 너무 어려워요.",                    # 공감 상황
    "회원 탈퇴는 어떻게 해요?",                # 안내 상황(학습셋에 없음)
]

def compare(question: str):
    before = chat(base_model, SYSTEM, question, temperature=0.7)
    after  = chat(dpo_model,  SYSTEM, question, temperature=0.7)
    print("=" * 70)
    print(f"❓ 질문: {question}")
    print(f"\n[학습 전] {before}")
    print(f"\n[학습 후] {after}")

for q in test_questions:
    compare(q)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


❓ 질문: 지금 당장 환불 안 되면 신고할 거예요.

[학습 전] 물론이죠! 당신의 의견에 따라대로 하겠습니다. 어떻게 도와드릴까요?

[학습 후] 그래 보이군요! 그럼 먼저 저희가 환불을 위해 최선을 다해 보겠습니다! 혹시 어떤 특정 제품이나 서비스에 대한 이야기였나요? 그 제품에 대해 좀 더 자세히 말씀해주실래요?
❓ 질문: 이 앱 너무 어려워요.

[학습 전] 별말해! 이 앱은 정말 쉽고 유용하죠! 가볍고 간단하게 사용하실 수 있을까요? 어떤 부분들이 좀 어렵나요?

[학습 후] 죄송합니다! 이 앱이 어렵다면 다른 사람이 도와드릴 수 있도록 다양한 옵션이 있습니다. 어떤 부분이 어려웠나요? 더 많은 지원 필요하시면 언제든 말씀해주세요!
❓ 질문: 회원 탈퇴는 어떻게 해요?

[학습 전] 안녕하세요! 회원 탈퇴를 원하시는 분이신가요? 저희 사이트에서 탈퇴하기 위해서는 아래의 단계들을 따라가시면 됩니다:

1. **로그인**: 우선, 자신의 계정을 로그인해주세요.
2. **프로필 관리**: 로그인 후, 마이페이지 또는 프로필 페이지에서 '탈퇴' 버튼을 찾아보세요.
3. **확인**: 이전에 작성하신 모든 메시지를 확인하고, 당신이 원하는 대로 선택하시면 '탈퇴' 버튼을 눌러주세요.

이후 7일간은 신규 가입이 불가능합니다. 이후에는 회원으로 다시 가입하실 수 있습니다. 

이제부터는 다른 사람과 소통할 때도 더 나아진 인상으로 전환되길 바랍니다~!

[학습 후] 네, 회원 탈퇴를 원하시면 쉽사리 할 수 있어요! 

1. **로그인** 후, 메뉴바에서 "마이페이지"를 선택하세요.
2. 마이페이지에서 "회원 관리"를 클릭합니다.
3. "탈퇴하기" 버튼을 눌러주세요.

그럼요, 정말 간단해요! 이렇게 하면 바로 탈퇴가 완료되답니다. 가까워요!


> 💡 학습 후 답이 더 **정중하고('~요'체), 거절 시 사유·대안을 곁들이는** 쪽으로 바뀌면 정렬 성공입니다.
> 변화가 약하면 ③의 `max_steps` 를 늘리거나 선호 쌍을 더 추가하세요.
> 반대로 답이 **반복·붕괴**하면 `beta` 를 키우거나(보수적) step을 줄이세요(과정렬 신호).

---

## ✏️ 연습문제

### 문제 1 — 새 페르소나 쌍 추가
`pairs` 에 **"이모지 남발 금지"** 를 가르치는 쌍을 2개 추가하세요.
(chosen=차분히 한 문장, rejected=과한 감탄사·이모지 남발). 내용은 비슷하게, **말투 한 축만** 다르게 하세요.
다시 ①~③을 돌려 학습 후 답에서 이모지가 줄었는지 ④로 확인하세요.

### 문제 2 — beta 비교 실험
③의 `beta` 를 **0.1 → 0.5** 로 바꿔 다시 학습하고, ④로 학습 전/후를 비교하세요.
beta가 클 때 말투 변화가 더 큰지, 혹은 답이 망가지는(반복/붕괴) 조짐이 있는지 관찰하세요.
(힌트: beta는 '참조모델에서 벗어날 정도'의 고삐입니다.)

---

## ✅ 해답

### 해답 1 — 이모지 절제 쌍 추가

In [18]:
extra_pairs = [
    {
        "prompt": "드디어 합격했어요!",
        "chosen": "우와, 정말 축하드려요! 그동안 고생하신 만큼 멋진 결과가 나와서 저도 너무 기뻐요. 앞으로의 길도 팍팍 응원할게요!",
        "rejected": "헐 대박!!!! 🎉🎉🎉 축하축하!!! 짱이에요 ㅠㅠㅠ 👏👏👏",
    },
    {
        "prompt": "오늘 발표 망친 것 같아요.",
        "chosen": "앗, 많이 속상하셨죠! 그래도 이번 한 번으로 모든 게 결정되는 건 절대 아니니까 너무 자책하지 마세요. 훌훌 털고 다음번에 더 멋지게 해내실 수 있을 거예요!",
        "rejected": "헐 ㅠㅠ 괜찮아요!!! 화이팅!!!! 다음엔 잘할거예요 😭😭😭🔥🔥",
    },
    {
        "prompt": "회사 가기 너무 싫어요.",
        "chosen": "아이고, 오늘따라 출근길이 더 무겁게 느껴지시나 봐요! 달달한 간식 하나 드시면서 오늘 하루도 기분 좋게 넘겨봐요. 제가 듬뿍 응원할게요!",
        "rejected": "아 ㅠㅠ 완전 공감!! 출근 진짜 극혐이죠 ㅠㅠ 퇴사 마려움 🔥",
    },
    {
        "prompt": "다이어트 중인데 자꾸 야식이 땡겨요.",
        "chosen": "다이어트 중에 참는 거 진짜 힘들죠! 그래도 꾹 참고 시원한 물 한 잔 먼저 드셔보시면 어떨까요? 내일 아침에 가뿐한 모습 보면 완전 뿌듯하실 거예요!",
        "rejected": "헐 ㅠㅠ 야식은 못 참죠!! 치킨 ㄱㄱ!! 다이어트는 내일부터!! 🍗🍗",
    },
    {
        "prompt": "요즘 잠을 잘 못 자서 너무 피곤해요.",
        "chosen": "아유, 많이 피곤하시겠어요! 오늘은 따뜻한 차 한 잔 드시고 일찍 푹 쉬면서 컨디션 싹 끌어올리셨으면 좋겠어요. 오늘 밤은 꼭 꿀잠 주무시길 바랄게요!",
        "rejected": "ㅠㅠ 헐 피곤하시겠다 ㅠㅠ 진짜 쓰러지는 거 아님?! 커피 수혈 시급!! ☕️☕️",
    }
]

# 기존 데이터에 합쳐 다시 빌드 → format_row 적용 후 ③ 재학습
from datasets import Dataset
rows = (pairs + extra_pairs) * 6
pref_ds = Dataset.from_list(rows).map(format_row)
print("추가 후 행 수:", len(pref_ds))
# (이후 ③ DPOTrainer 셀을 다시 실행 → ④로 이모지가 줄었는지 비교)

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

추가 후 행 수: 78


> 포인트: chosen/rejected가 **'이모지·감탄사' 한 축만** 다릅니다. 그래야 모델이 '절제된 톤'을 정확히 학습합니다.

### 해답 2 — beta 비교

In [19]:
# ③의 DPOConfig 에서 beta 만 바꿔 다시 학습하는 함수
def train_with_beta(beta_value, steps=60):
    from trl import DPOTrainer, DPOConfig
    cfg = DPOConfig(
        output_dir=f"{WORK}/dpo_beta_{beta_value}",
        beta=beta_value, learning_rate=5e-5,
        per_device_train_batch_size=1, gradient_accumulation_steps=4,
        max_steps=steps, logging_steps=5,
        max_length=512, report_to="none", fp16=False, bf16=False,
    )
    tr = DPOTrainer(model=base_model, args=cfg, train_dataset=pref_ds,
                    processing_class=tok, peft_config=lora_cfg)
    tr.train()
    return tr.model

# beta=0.5 로 재학습 후 비교 (주의: base_model 을 다시 쓰므로 런타임 메모리 여유 확인)
dpo_model_b05 = train_with_beta(0.5)
dpo_model_b05.eval()
print("\n[beta=0.5] 결과:")
print(chat(dpo_model_b05, SYSTEM, "지금 당장 환불 안 되면 신고할 거예요."))

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

Step,Training Loss
5,0.506250
10,0.094327
15,0.004642
20,0.000225
25,0.000107
30,0.000072
35,0.000028
40,0.000028
45,0.000035
50,0.000023



[beta=0.5] 결과:


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


물론이죠! 당신의 의견에 따라 조정하겠습니다. 어떻게 도와드릴까요?


> 포인트: beta가 크면 변화가 과감해지지만, 너무 크면 **과정렬**로 답이 반복·붕괴할 수 있습니다.
> 정량(margins)과 정성(샘플)을 **함께** 보고 적절한 beta를 고르세요.

---

## 🚀 직접 써보는 데모 — 학습 전/후 나란히 비교 (Gradio)

질문을 입력하면 **학습 전(원본)** 과 **학습 후(DPO 정렬)** 모델의 답을 **나란히** 보여 줍니다.
같은 질문에 말투·거절 스타일이 어떻게 달라졌는지 한눈에 비교할 수 있습니다. 위에서 만든 `chat()` 을 그대로 호출합니다.

In [20]:
# [첫 셀에서 일괄 설치] %pip install -q gradio
import gradio as gr

def demo_compare(question: str):
    question = (question or "").strip()
    if not question:
        return "(질문을 입력하세요)", "(질문을 입력하세요)"
    try:
        before = chat(base_model, SYSTEM, question, temperature=0.7)
        after  = chat(dpo_model,  SYSTEM, question, temperature=0.7)
    except Exception as e:                       # 생성 중 에러도 화면에 안전하게 표시
        return f"(에러) {e}", f"(에러) {e}"
    return before, after

with gr.Blocks(title="DPO 선호 정렬 — 학습 전/후 비교") as demo:
    gr.Markdown("## 🪄 DPO 선호 정렬 — 같은 질문, 학습 전 vs 후 말투 비교")
    gr.Markdown(f"**페르소나:** {SYSTEM}")
    with gr.Row():
        q = gr.Textbox(label="질문", lines=2,
                       placeholder="예: 지금 당장 환불 안 되면 신고할 거예요.")
    btn = gr.Button("두 모델 비교", variant="primary")
    gr.Examples(
        examples=[
            "지금 당장 환불 안 되면 신고할 거예요.",
            "이 앱 너무 어려워요.",
            "회원 탈퇴는 어떻게 해요?",
            "주말에 상담 가능해요?",
        ],
        inputs=q,
    )
    with gr.Row():
        out_before = gr.Textbox(label="🟦 학습 전 (원본 모델)", lines=8)
        out_after  = gr.Textbox(label="🟩 학습 후 (DPO 정렬)", lines=8)
    btn.click(fn=demo_compare, inputs=q, outputs=[out_before, out_after])

# share=True 로 외부 공유 가능한 임시 링크 생성 (코랩/캐글에서도 동작)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

Keyboard interruption in main thread... closing server.


> 💡 질문을 넣고 **두 모델 비교**를 누르면, 왼쪽(원본)과 오른쪽(정렬됨)의 답이 함께 나타납니다.
> 거절 상황에서 오른쪽이 더 **정중하고 대안을 곁들이는지**, 일상 질문에서 더 **따뜻한 '~요'체**인지 확인하세요.
> (연습문제로 이모지 절제 쌍을 추가했다면, 감탄사·이모지 차이도 눈에 보입니다.)

---

## 🧾 한 장 정리
- **정렬(alignment)** = "무엇을 아느냐"가 아니라 **"어떻게 답하느냐"** 를 사람의 선호에 맞추기.
- **SFT의 한계**(정답 하나·나쁜 답 못 가르침) → **선호 쌍**(prompt·chosen·rejected)으로 좋은답↑ 나쁜답↓.
- **DPO** = 보상모델·강화학습 **없이** 선호로 직접 최적화 (RLHF/PPO보다 가볍고 안정적). 손실은 `-log_sigmoid(beta·(좋은답_여유 − 나쁜답_여유))`.
- **TRL DPOTrainer + LoRA** = 데이터 칼럼만 `prompt/chosen/rejected` 로 맞추면 참조모델·손실 자동 처리. T4에서도 동작.
- **검증이 곧 정렬.** 학습 전/후를 **정량(margins) + 정성(샘플)** 으로 비교 — 결만 바뀌고 능력은 보존됐는지 확인.